# ⚡ Evolve — AI Code Evolution Engine

**Evolutionary optimization for Python code using Claude AI.**

This notebook demonstrates the full Evolve pipeline:
1. You paste a slow Python function
2. Claude generates optimized variants using 6 different strategies
3. Each variant is **actually executed and benchmarked** (real timing!)
4. The best variants become "parents" for the next generation
5. After N generations, you get the fastest correct version

---

### 🔑 Setup (one time only)
Before running, add your Anthropic API key to **Colab Secrets**:
1. Click the 🔑 **Secrets** icon in the left sidebar
2. Click **+ Add new secret**
3. Name: `ANTHROPIC_API_KEY`
4. Value: paste your key (starts with `sk-ant-...`)
5. Toggle **Notebook access** ON

Then press **Runtime → Run all** and watch it go!


## 1. Setup

In [ ]:
# Install Anthropic SDK
!pip install anthropic tabulate -q
print('✅ Dependencies installed')

In [ ]:
# Load API key from Colab Secrets
from google.colab import userdata

try:
    API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    # Fallback: ask for manual input
    from getpass import getpass
    API_KEY = getpass('Anthropic API key not found in Secrets. Enter it manually: ')

MODEL = 'claude-3-5-sonnet-20241022'
print(f'✅ Using model: {MODEL}')

## 2. The Evolution Engine

### How it works

The engine uses **6 optimization strategies**, each targeting a different performance angle:

| Strategy | What it does |
|---|---|
| **Algorithmic** | Better data structures, reduced Big-O complexity |
| **Vectorized** | NumPy array operations instead of Python loops |
| **Built-in** | Python stdlib (collections, itertools, functools) |
| **Memory** | Generators, cache locality, minimal allocations |
| **Parallel** | multiprocessing/threading for CPU/IO-bound work |
| **Cython-style** | Type hints, local vars, struct — pure Python but C-fast |

Each generation:
1. Claude generates one variant per strategy
2. Each variant is **executed** with test inputs and **timed**
3. Correctness is verified by comparing outputs to the original
4. The best correct variant becomes the "parent" for the next generation


In [ ]:
import time
import json
import traceback
import textwrap
import numpy as np
import anthropic
from tabulate import tabulate
from typing import Any, Callable
from dataclasses import dataclass, field
from IPython.display import display, HTML, clear_output

client = anthropic.Anthropic(api_key=API_KEY)

# ─── Optimization Strategies ─────────────────────────────────────────────────

STRATEGIES = [
    {
        'name': 'algorithmic',
        'prompt': 'Optimize this Python function by improving its algorithmic complexity. '
                  'Use better data structures (sets for lookups, heaps for priority queues), '
                  'reduce nested loops, apply dynamic programming or memoization. '
                  'Focus on reducing Big-O complexity.',
    },
    {
        'name': 'vectorized',
        'prompt': 'Optimize this Python function using NumPy vectorization. '
                  'Replace Python loops with NumPy array operations, use broadcasting, '
                  'and leverage built-in NumPy functions. Same inputs/outputs.',
    },
    {
        'name': 'builtin',
        'prompt': 'Optimize using Python built-in functions and standard library. '
                  'Use map/filter, itertools, functools, collections.Counter, bisect, etc. '
                  'Replace manual implementations with C-level built-ins.',
    },
    {
        'name': 'memory',
        'prompt': 'Optimize for memory efficiency and cache performance. '
                  'Use generators, minimize object creation, optimize data layout. '
                  'Keep the same function signature and outputs.',
    },
    {
        'name': 'parallel',
        'prompt': 'Optimize using concurrent execution. Use multiprocessing.Pool '
                  'for CPU-bound work. Ensure same signature and identical results.',
    },
    {
        'name': 'cython_style',
        'prompt': 'Rewrite in a style that would be fast if compiled — type hints, '
                  'local variables, avoiding dynamic dispatch. Pure Python but C-speed.',
    },
]

SYSTEM_PROMPT = '''You are an expert Python performance engineer.
Your job is to optimize Python code for maximum speed while maintaining correctness.

RULES:
1. Return ONLY the optimized Python function. No explanations, no markdown, no code fences.
2. The function must have the EXACT same name and parameters as the original.
3. The function must produce IDENTICAL outputs for all inputs.
4. You may add imports at the top if needed (numpy, itertools, etc).
5. Do NOT use packages outside the standard library or numpy.
6. Focus on MEASURABLE speedup, not cosmetic changes.'''

print('✅ Evolution engine loaded')

### Core functions: generate, benchmark, validate

In [ ]:
@dataclass
class Variant:
    code: str
    strategy: str
    speedup: float = 0.0
    is_correct: bool = False
    original_time: float = 0.0
    optimized_time: float = 0.0
    error: str = ''


def generate_variant(original_code: str, strategy: dict,
                     parent_code: str = None, feedback: str = None) -> str:
    """Ask Claude to generate an optimized variant."""
    user_msg = f"{strategy['prompt']}\n\nORIGINAL CODE:\n{original_code}"
    if parent_code:
        user_msg += f"\n\nPREVIOUS BEST (improve upon this):\n{parent_code}"
    if feedback:
        user_msg += f"\n\nFEEDBACK: {feedback}"

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            system=SYSTEM_PROMPT,
            messages=[
                {'role': 'user', 'content': user_msg},
            ],
        )
    except Exception as e:
        raise RuntimeError(f'Claude API error: {type(e).__name__}: {e}')
    code = response.content[0].text.strip()
    # Strip markdown fences if present
    code = code.replace('```python', '').replace('```', '').strip()
    return code


def extract_function(code: str, func_name: str):
    """Execute code string and extract the named function."""
    namespace = {}
    exec(code, namespace)
    if func_name not in namespace:
        raise ValueError(f"Function '{func_name}' not found in generated code")
    return namespace[func_name]


def benchmark_function(func: Callable, test_inputs: list,
                       n_runs: int = 5) -> float:
    """Benchmark a function and return median execution time in seconds."""
    # Warmup
    for args in test_inputs:
        func(*args)

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        for args in test_inputs:
            func(*args)
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    return float(np.median(times))


def validate_correctness(original_func: Callable, optimized_func: Callable,
                         test_inputs: list) -> tuple:
    """Check that optimized function produces identical outputs."""
    for i, args in enumerate(test_inputs):
        try:
            orig_result = original_func(*args)
            opt_result = optimized_func(*args)

            # Handle different result types
            if isinstance(orig_result, np.ndarray):
                if not np.allclose(orig_result, np.array(opt_result), rtol=1e-5):
                    return False, f'Output mismatch on test case {i}'
            elif isinstance(orig_result, float):
                if abs(orig_result - opt_result) > 1e-6:
                    return False, f'Float mismatch on test case {i}: {orig_result} vs {opt_result}'
            elif isinstance(orig_result, (list, tuple)):
                try:
                    if sorted(orig_result) != sorted(opt_result):
                        if list(orig_result) != list(opt_result):
                            return False, f'List mismatch on test case {i}'
                except TypeError:
                    if list(orig_result) != list(opt_result):
                        return False, f'List mismatch on test case {i}'
            else:
                if orig_result != opt_result:
                    return False, f'Output mismatch on test case {i}: {orig_result} vs {opt_result}'
        except Exception as e:
            return False, f'Error on test case {i}: {str(e)}'

    return True, 'All test cases passed'


print('✅ Core functions defined')

### The Evolution Loop

In [ ]:
def evolve(original_code: str, func_name: str, test_inputs: list,
           max_generations: int = 3, strategies_per_gen: int = 4,
           target_speedup: float = 50.0) -> list:
    """
    Run the full evolutionary optimization loop.

    Args:
        original_code: The Python source code to optimize
        func_name: Name of the function to optimize
        test_inputs: List of (args_tuple) to test with
        max_generations: How many evolution cycles
        strategies_per_gen: How many strategies to try per generation
        target_speedup: Stop early if this speedup is achieved

    Returns:
        List of all variants tested, sorted by speedup (best first)
    """
    # Extract and benchmark original
    original_func = extract_function(original_code, func_name)
    original_time = benchmark_function(original_func, test_inputs)
    print(f'📊 Original function: {original_time*1000:.2f}ms')
    print(f'   Running {max_generations} generations × {strategies_per_gen} strategies')
    print('=' * 60)

    all_variants = []
    best_parent_code = None
    best_speedup = 1.0

    for gen in range(1, max_generations + 1):
        print(f'\n🧬 Generation {gen}/{max_generations}')
        print('-' * 40)

        selected_strategies = STRATEGIES[:strategies_per_gen]

        for strat in selected_strategies:
            variant = Variant(code='', strategy=strat['name'])
            try:
                # Generate
                print(f"  ⚙️  {strat['name']:15s} ", end='', flush=True)
                variant.code = generate_variant(
                    original_code, strat,
                    parent_code=best_parent_code,
                    feedback=f'Previous best: {best_speedup:.1f}x. Beat it.' if gen > 1 else None
                )

                # Extract & validate
                opt_func = extract_function(variant.code, func_name)
                is_correct, details = validate_correctness(original_func, opt_func, test_inputs)
                variant.is_correct = is_correct

                if is_correct:
                    # Benchmark
                    opt_time = benchmark_function(opt_func, test_inputs)
                    variant.original_time = original_time
                    variant.optimized_time = opt_time
                    variant.speedup = original_time / opt_time if opt_time > 0 else 0
                    print(f'→ ✅ {variant.speedup:8.1f}x  ({opt_time*1000:.2f}ms)')
                else:
                    variant.error = details
                    print(f'→ ❌ incorrect ({details[:50]})')

            except Exception as e:
                variant.error = str(e)
                print(f'→ 💥 error ({str(e)[:50]})')

            all_variants.append(variant)

        # Update best parent
        correct_variants = [v for v in all_variants if v.is_correct and v.speedup > best_speedup]
        if correct_variants:
            best = max(correct_variants, key=lambda v: v.speedup)
            best_speedup = best.speedup
            best_parent_code = best.code
            print(f'\n  🏆 Best so far: {best_speedup:.1f}x ({best.strategy})')

        if best_speedup >= target_speedup:
            print(f'\n🎯 Target speedup {target_speedup}x reached! Stopping early.')
            break

    # Sort by speedup
    all_variants.sort(key=lambda v: v.speedup if v.is_correct else 0, reverse=True)
    return all_variants


print('✅ Evolution loop ready')

### Results visualization

In [ ]:
def show_results(variants: list, original_code: str):
    """Display a rich summary of evolution results."""
    correct = [v for v in variants if v.is_correct and v.speedup > 1]
    incorrect = [v for v in variants if not v.is_correct]

    print('\n' + '=' * 60)
    print('📊 EVOLUTION RESULTS')
    print('=' * 60)

    # Summary table
    table_data = []
    for v in variants:
        status = '✅' if v.is_correct else '❌'
        speedup = f'{v.speedup:.1f}x' if v.is_correct and v.speedup > 0 else 'N/A'
        time_ms = f'{v.optimized_time*1000:.2f}ms' if v.optimized_time > 0 else 'N/A'
        table_data.append([status, v.strategy, speedup, time_ms, v.error[:40] if v.error else ''])

    print(tabulate(table_data,
                   headers=['', 'Strategy', 'Speedup', 'Time', 'Notes'],
                   tablefmt='rounded_grid'))

    print(f'\n📈 Total variants tested: {len(variants)}')
    print(f'   Correct & faster: {len(correct)}')
    print(f'   Incorrect: {len(incorrect)}')

    if correct:
        best = correct[0]
        print(f'\n🏆 BEST RESULT: {best.speedup:.1f}x faster ({best.strategy})')
        print(f'   Original:  {best.original_time*1000:.2f}ms')
        print(f'   Optimized: {best.optimized_time*1000:.2f}ms')
        print(f'\n📝 Optimized code:')
        print('-' * 60)
        print(best.code)
        print('-' * 60)
    else:
        print('\n⚠️ No correct optimizations found. Try more generations.')


print('✅ Visualization ready')

## 3. Example: Optimize `find_duplicates`

A classic O(n²) nested loop that finds duplicate elements in a list.
Let's see how much Claude can speed it up with **real benchmarks**.


In [ ]:
# The slow function to optimize
ORIGINAL_CODE = '''
def find_duplicates(lst):
    """Find all duplicate elements in a list."""
    duplicates = []
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j] and lst[i] not in duplicates:
                duplicates.append(lst[i])
    return duplicates
'''.strip()

# Test inputs — from small to large
import random
random.seed(42)
TEST_INPUTS = [
    ([1, 2, 3, 2, 4, 5, 3, 6],),
    (list(range(100)) + list(range(50)),),
    ([random.randint(0, 500) for _ in range(2000)],),
    ([random.randint(0, 1000) for _ in range(5000)],),
]

print('Original code:')
print(ORIGINAL_CODE)
print(f'\nTest inputs: {len(TEST_INPUTS)} cases (up to 5000 elements)')

In [ ]:
# 🚀 Run the evolution!
variants = evolve(
    original_code=ORIGINAL_CODE,
    func_name='find_duplicates',
    test_inputs=TEST_INPUTS,
    max_generations=3,
    strategies_per_gen=4,
    target_speedup=100.0,
)

show_results(variants, ORIGINAL_CODE)

## 4. Example: Optimize `find_primes`

Trial division O(n²) vs Sieve of Eratosthenes. Can Claude figure it out?


In [ ]:
PRIMES_CODE = '''
def find_primes(n):
    """Find all primes up to n."""
    primes = []
    for num in range(2, n):
        is_prime = True
        for i in range(2, num):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes
'''.strip()

PRIMES_INPUTS = [
    (100,),
    (1000,),
    (5000,),
    (10000,),
]

print('Original code:')
print(PRIMES_CODE)

In [ ]:
# 🚀 Evolve the prime finder!
prime_variants = evolve(
    original_code=PRIMES_CODE,
    func_name='find_primes',
    test_inputs=PRIMES_INPUTS,
    max_generations=3,
    strategies_per_gen=4,
    target_speedup=200.0,
)

show_results(prime_variants, PRIMES_CODE)

## 5. 🔧 Try Your Own Code!

Paste your own slow Python function below and let Evolve optimize it.

**Instructions:**
1. Replace `YOUR_CODE` with your function
2. Set `FUNC_NAME` to the function name
3. Add test inputs as a list of tuples (each tuple = one set of arguments)
4. Run the cell!


In [ ]:
# ✏️ PASTE YOUR CODE HERE
YOUR_CODE = '''
def my_function(data):
    """Replace this with your slow function."""
    result = []
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] + data[j] == 0:
                result.append((data[i], data[j]))
    return result
'''.strip()

FUNC_NAME = 'my_function'

YOUR_TEST_INPUTS = [
    ([1, -1, 2, -2, 3, 0, 5, -5],),
    (list(range(-100, 100)),),
    ([i * (-1)**i for i in range(500)],),
]

# 🚀 Run evolution on YOUR code
my_variants = evolve(
    original_code=YOUR_CODE,
    func_name=FUNC_NAME,
    test_inputs=YOUR_TEST_INPUTS,
    max_generations=3,
    strategies_per_gen=6,  # Try all 6 strategies!
)

show_results(my_variants, YOUR_CODE)